# Chapitre 14 - Exercices 14.22 à 14.25

Ce notebook traite les derniers exercices du chapitre :

- **14.22** : PageRank (normalisation et méthode de la puissance) ;
- **14.23** : l'algorithme de factorisation en matrices non négatives (NMF) vu comme un
  algorithme MM (minoration–maximisation) ;
- **14.24** : NMF de rang 1 et ajustement proportionnel itératif ;
- **14.25** : NMF contre ACP sur les chiffres « 2 » manuscrits.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import ch14_non_supervise_complements as nsc
from esl import datasets
set_style()

## Exercice 14.22 : PageRank

**Énoncé (reformulé).**

(a) Montrer que la définition (14.108) implique que la somme des PageRanks $p_i$ vaut
$N$, le nombre de pages web.

(b) Écrire un programme calculant les PageRanks par la méthode de la puissance à partir
de la formulation (14.107), et l'appliquer au réseau de la figure 14.47 du livre.

### Rappels (section 14.10)

On note $L_{ij} = 1$ si la page $j$ pointe vers la page $i$ (0 sinon), et
$c_j = \sum_iL_{ij}$ le nombre de liens sortants de la page $j$. Les PageRanks sont
définis par la relation récursive (14.107) :
$$p_i = (1 - d) + d\sum_{j=1}^N\frac{L_{ij}}{c_j}\,p_j,$$
avec $d = 0{,}85$ ; matriciellement (14.108) :
$$p = (1-d)\,e + d\,LD_c^{-1}p,\qquad e = (1,\dots,1)^T,\quad D_c = \operatorname{diag}(c).$$
Chaque page $j$ distribue un « vote » total égal à $p_j$, réparti également entre les
$c_j$ pages vers lesquelles elle pointe.

### (a) La somme des PageRanks vaut $N$

On suppose que chaque page a au moins un lien sortant ($c_j \ge 1$, sinon $D_c^{-1}$
n'est pas défini). La somme de la colonne $j$ de $LD_c^{-1}$ vaut
$\sum_iL_{ij}/c_j = c_j/c_j = 1$, c'est-à-dire $e^TLD_c^{-1} = e^T$.

Multiplions (14.108) à gauche par $e^T$ :
$$e^Tp = (1-d)\,e^Te + d\,e^TLD_c^{-1}p = (1-d)N + d\,e^Tp.$$
Donc $(1 - d)\,e^Tp = (1-d)N$, et comme $d < 1$ :
$$\boxed{\sum_{i=1}^Np_i = e^Tp = N.}$$
La moyenne des PageRanks vaut 1 ; la normalisation $e^Tp = N$ utilisée pour écrire
(14.109) n'est donc pas une contrainte supplémentaire, elle découle de la définition.

### Le point de vue « chaîne de Markov »

Grâce à (a), on peut écrire $(1-d)e = (1-d)ee^Tp/N$, d'où $p = Ap$ avec
$$A = (1-d)\frac{ee^T}{N} + d\,LD_c^{-1}.\qquad(14.109)$$
Les coefficients de $A$ sont positifs et ses colonnes somment à 1 : $A$ est la matrice
de transition d'une chaîne de Markov (le **surfeur aléatoire** : depuis la page $j$, il
suit un lien sortant au hasard avec probabilité $d$, ou saute sur une page uniforme avec
probabilité $1-d$). $p/N$ est la loi stationnaire de cette chaîne, et la valeur propre 1
est la plus grande (en module) de $A$ : la **méthode de la puissance**
$p_k \leftarrow Ap_{k-1}$, $p_k \leftarrow Np_k/e^Tp_k$ (14.110) converge vers $p$.

### (b) Programme et vérification sur l'exemple du livre

La fonction `pagerank_power` implémente (14.110). On vérifie d'abord l'exemple de la
figure 14.46 (matrice (14.111)), pour lequel le livre donne
$\hat p = (1{,}49;\ 0{,}78;\ 1{,}58;\ 0{,}15)$.